# T1.5 — EDA on HAI Bronze

Reads the Bronze Parquet layer (from `make ingest`) and gathers the facts needed
to write the Silver/Gold Pandera schema in T2. Runs on both Windows/WSL2 and Mac
because every path resolves through `PathConfig` — nothing is hardcoded.

Prereq: Bronze must exist locally. If not, run: `make ingest`

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from cps_anomaly_pipeline.paths import PathConfig
from cps_anomaly_pipeline.ingest import LABEL_COLUMNS

# Notebook CWD is notebooks/; point DATA_ROOT at the repo-root data/ dir
# unless DATA_ROOT is already set in the environment.
if "DATA_ROOT" not in os.environ:
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.environ["DATA_ROOT"] = str(repo_root / "data")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

paths = PathConfig()
bronze = paths.hai_bronze_dir
print("Bronze dir:", bronze)

parquets = sorted(bronze.glob("*.parquet"))
print(f"{len(parquets)} Bronze files:")
for p in parquets:
    print(" ", p.name)
assert parquets, "No Bronze parquet found. Run `make ingest` first."

## 1. Load train (normal) vs test (labelled) separately

Train files carry no attack labels; test files carry `attack` + `attack_P1/P2/P3`.

In [ ]:
train_files = [p for p in parquets if p.name.startswith("train")]
test_files  = [p for p in parquets if p.name.startswith("test")]

train = pd.concat([pd.read_parquet(p) for p in train_files], ignore_index=True)
test  = pd.concat([pd.read_parquet(p) for p in test_files],  ignore_index=True)

print("train shape:", train.shape)
print("test  shape:", test.shape)

## 2. Column inventory

Separate provenance, label, and sensor/actuator columns. Sensor columns are the
modelling features. Confirm `attack_P4` is absent (HAI 21.03 labels P1-P3 only).

In [ ]:
PROVENANCE = ["source_file", "ingested_at"]
labels = [c for c in LABEL_COLUMNS if c in test.columns]

sensor_cols = [c for c in train.columns if c not in PROVENANCE and c not in LABEL_COLUMNS]
print(f"sensor/actuator columns: {len(sensor_cols)}")
print(f"label columns present in test: {labels}")
print("attack_P4 present?", "attack_P4" in test.columns)

for stage in ["P1_", "P2_", "P3_", "P4_"]:
    n = sum(c.startswith(stage) for c in sensor_cols)
    print(f"  {stage[:-1]}: {n} columns")

## 3. Dtypes and per-column summary

Which columns are continuous floats vs discrete/binary actuators. Actuators with
a tiny unique-value count are candidates for discrete handling in Silver.

In [ ]:
summary = pd.DataFrame({
    "dtype": train[sensor_cols].dtypes.astype(str),
    "n_unique": train[sensor_cols].nunique(),
    "min": train[sensor_cols].min(numeric_only=True),
    "max": train[sensor_cols].max(numeric_only=True),
    "std": train[sensor_cols].std(numeric_only=True),
})
summary = summary.sort_values("n_unique")
summary.head(30)

## 4. Constant columns (variance = 0)

Zero-variance sensors carry no information — flag them for removal in Silver.
A column constant in train but varying in test may itself signal an attack, so
note those separately rather than dropping blindly.

In [ ]:
const_train = [c for c in sensor_cols if train[c].nunique(dropna=False) <= 1]
print(f"Constant in TRAIN: {len(const_train)}")
print(const_train)

const_but_varies_in_test = [
    c for c in const_train
    if c in test.columns and test[c].nunique(dropna=False) > 1
]
print("\nConstant in train but VARIES in test (do NOT drop blindly):")
print(const_but_varies_in_test)

## 5. Missing values, attack ratio, distribution shift

In [ ]:
# --- NaN ---
na_train = train[sensor_cols].isna().sum()
na_test  = test[sensor_cols].isna().sum()
print("Columns with NaN in train:", int((na_train > 0).sum()))
print("Columns with NaN in test :", int((na_test > 0).sum()))
print(na_train[na_train > 0].sort_values(ascending=False).head(10))

# --- Attack ratio in test ---
print("\n--- attack ratio (test) ---")
for c in labels:
    print(f"{c:12s} ratio={test[c].mean():.4%}  counts={test[c].value_counts().to_dict()}")

# --- Distribution shift: train (normal) vs test ---
num_cols = [c for c in sensor_cols if pd.api.types.is_numeric_dtype(train[c])]
eps = 1e-9
shift = pd.DataFrame({
    "train_mean": train[num_cols].mean(),
    "test_mean":  test[num_cols].mean(),
    "train_std":  train[num_cols].std(),
})
shift["z_shift"] = (shift["test_mean"] - shift["train_mean"]).abs() / (shift["train_std"] + eps)
shift = shift.sort_values("z_shift", ascending=False)
print("\n--- top distribution shifts (train normal vs test) ---")
print(shift.head(15))

In [ ]:
top = shift.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), top):
    ax.hist(train[col].dropna(), bins=50, alpha=0.6, label="train (normal)", density=True)
    ax.hist(test[col].dropna(),  bins=50, alpha=0.6, label="test", density=True)
    ax.set_title(f"{col}  (z_shift={shift.loc[col, 'z_shift']:.2f})")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()